# 🔗 Séance 4 – Croiser deux tables et visualiser les données
**SNT – Thème : Les données structurées et leur traitement**

---

## 🎯 Objectifs
- Comprendre pourquoi on utilise **plusieurs tables** liées entre elles
- Réaliser un **croisement de tables** sur un descripteur commun
- Créer des **visualisations graphiques** à partir des données

---

## ⏱️ Durée : 1h30


---
## Partie 1 – Pourquoi plusieurs tables ? *(20 min)*

### ❌ Le problème d'une seule grande table

Imaginons qu'une bibliothèque gère ses données avec UNE SEULE table :

| id_emprunt | livre_titre | livre_auteur | livre_isbn | abonne_nom | abonne_prenom | abonne_email | date_emprunt |
|---|---|---|---|---|---|---|---|
| 1 | Le Petit Prince | Saint-Exupéry | 978-2070408504 | Dupont | Léa | lea@mail.fr | 2024-01-10 |
| 2 | 1984 | George Orwell | 978-2070368228 | Dupont | Léa | lea@mail.fr | 2024-01-15 |
| 3 | Le Petit Prince | Saint-Exupéry | 978-2070408504 | Martin | Tom | tom@mail.fr | 2024-01-20 |

### ✏️ Exercice 1 – Identifier les problèmes

1. Léa Dupont change d'adresse email. Combien de lignes faut-il modifier ?
2. Si on se trompe dans l'orthographe de « Saint-Exupéry » à la ligne 1, que se passe-t-il ?
3. Si on supprime la ligne 3, perd-on des informations sur Léa Dupont ?
4. Combien de fois l'information `978-2070408504` est-elle répétée inutilement ?

👉 Ces problèmes s'appellent des **redondances** et des **anomalies**. La solution : diviser en **plusieurs tables liées**.

---
### ✅ La solution : 3 tables reliées

On sépare les informations en 3 tables :
- **Table Livres** : infos sur les livres
- **Table Abonnés** : infos sur les abonnés
- **Table Emprunts** : qui a emprunté quoi et quand (avec des identifiants)

Chaque table possède un **identifiant unique** (clé) qui permet de relier les tables entre elles.

In [ ]:
# Création des 3 tables de la bibliothèque
import csv

# Table 1 : les livres
livres_data = [
    ["id_livre", "titre", "auteur", "genre", "annee"],
    ["L001", "Le Petit Prince", "Antoine de Saint-Exupery", "Conte", 1943],
    ["L002", "1984", "George Orwell", "Dystopie", 1949],
    ["L003", "Harry Potter T1", "J.K. Rowling", "Fantastique", 1997],
    ["L004", "L'Alchimiste", "Paulo Coelho", "Roman", 1988],
    ["L005", "Le Seigneur des Anneaux", "J.R.R. Tolkien", "Fantastique", 1954],
    ["L006", "Germinal", "Emile Zola", "Roman", 1885],
]

# Table 2 : les abonnés
abonnes_data = [
    ["id_abonne", "nom", "prenom", "email", "ville"],
    ["A001", "Dupont", "Lea", "lea@mail.fr", "Paris"],
    ["A002", "Martin", "Tom", "tom@mail.fr", "Lyon"],
    ["A003", "Bernard", "Jade", "jade@mail.fr", "Paris"],
    ["A004", "Petit", "Lucas", "lucas@mail.fr", "Bordeaux"],
]

# Table 3 : les emprunts (table de liaison)
emprunts_data = [
    ["id_emprunt", "id_livre", "id_abonne", "date_emprunt", "date_retour"],
    ["E001", "L001", "A001", "2024-01-10", "2024-01-24"],
    ["E002", "L002", "A001", "2024-01-15", "2024-01-29"],
    ["E003", "L001", "A002", "2024-01-20", "2024-02-03"],
    ["E004", "L003", "A003", "2024-02-01", "2024-02-15"],
    ["E005", "L004", "A002", "2024-02-05", "2024-02-19"],
    ["E006", "L005", "A004", "2024-02-10", "2024-02-24"],
    ["E007", "L002", "A003", "2024-02-12", "2024-02-26"],
    ["E008", "L006", "A001", "2024-03-01", "2024-03-15"],
    ["E009", "L003", "A002", "2024-03-05", "2024-03-19"],
]

for nom_fichier, donnees in [("livres.csv", livres_data), 
                              ("abonnes.csv", abonnes_data), 
                              ("emprunts.csv", emprunts_data)]:
    with open(nom_fichier, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(donnees)

print("Les 3 fichiers CSV ont été créés !")

In [ ]:
# Chargement des 3 tables
import csv

def charger_csv(nom_fichier):
    """Charge un CSV et retourne une liste de dictionnaires"""
    donnees = []
    with open(nom_fichier, encoding="utf-8") as f:
        for ligne in csv.DictReader(f):
            donnees.append(ligne)
    return donnees

livres = charger_csv("livres.csv")
abonnes = charger_csv("abonnes.csv")
emprunts = charger_csv("emprunts.csv")

print(f"{len(livres)} livres, {len(abonnes)} abonnés, {len(emprunts)} emprunts")

---
## Partie 2 – Croiser les tables *(30 min)*

Pour savoir **qui a emprunté quoi**, on doit croiser les 3 tables grâce aux identifiants communs.

In [ ]:
# Méthode : créer des dictionnaires indexés par id pour un accès rapide

# Dictionnaire des livres : { "L001": {dict livre}, "L002": {dict livre}, ... }
livres_index = {}
for livre in livres:
    livres_index[livre["id_livre"]] = livre

# Dictionnaire des abonnés
abonnes_index = {}
for abonne in abonnes:
    abonnes_index[abonne["id_abonne"]] = abonne

print("Index créés !")
print("Exemple d'accès au livre L001 :")
print(livres_index["L001"])

In [ ]:
# Croisement : afficher TOUS les emprunts avec noms et titres
print("=== LISTE COMPLÈTE DES EMPRUNTS ===")
print(f"{'Abonné':<20} {'Livre':<30} {'Date':<15}")
print("-" * 65)

for emprunt in emprunts:
    # On retrouve le livre grâce à son id
    livre = livres_index[emprunt["id_livre"]]
    # On retrouve l'abonné grâce à son id
    abonne = abonnes_index[emprunt["id_abonne"]]
    
    nom_complet = abonne["prenom"] + " " + abonne["nom"]
    print(f"{nom_complet:<20} {livre['titre']:<30} {emprunt['date_emprunt']:<15}")

In [ ]:
# Question : Quels livres Léa Dupont a-t-elle empruntés ?
abonne_cherche = "A001"  # id de Léa Dupont

abonne = abonnes_index[abonne_cherche]
print(f"Livres empruntés par {abonne['prenom']} {abonne['nom']} :")

for emprunt in emprunts:
    if emprunt["id_abonne"] == abonne_cherche:
        livre = livres_index[emprunt["id_livre"]]
        print(f"  - {livre['titre']} (le {emprunt['date_emprunt']})")

### ✏️ Exercice 2 – Requêtes croisées

In [ ]:
# Question 1 : Qui a emprunté le livre "1984" (L002) ?
# (affiche les noms et prénoms des abonnés)

# Ton code ici :


In [ ]:
# Question 2 : Quel livre a été emprunté le plus souvent ?
# AIDE : compte le nombre d'emprunts par id_livre

# Ton code ici :


In [ ]:
# Question 3 : Affiche tous les livres de genre "Fantastique" empruntés
# (croiser la table emprunts avec la table livres, filtrer sur le genre)

# Ton code ici :


---
## Partie 3 – Visualisation graphique *(25 min)*

Python peut créer des graphiques avec le module `matplotlib`.

In [ ]:
# Graphique 1 : Nombre d'emprunts par abonné
import matplotlib.pyplot as plt

# On compte les emprunts par abonné
compteur = {}
for emprunt in emprunts:
    id_ab = emprunt["id_abonne"]
    if id_ab not in compteur:
        compteur[id_ab] = 0
    compteur[id_ab] = compteur[id_ab] + 1

# On prépare les données pour le graphique
noms = []
nbr_emprunts = []
for id_ab, nb in compteur.items():
    abonne = abonnes_index[id_ab]
    noms.append(abonne["prenom"] + " " + abonne["nom"])
    nbr_emprunts.append(nb)

# Création du graphique en barres
plt.figure(figsize=(8, 5))
plt.bar(noms, nbr_emprunts, color=["steelblue", "coral", "mediumseagreen", "mediumpurple"])
plt.title("Nombre d'emprunts par abonné")
plt.xlabel("Abonné")
plt.ylabel("Nombre d'emprunts")
plt.tight_layout()
plt.savefig("graphique_emprunts.png")
plt.show()
print("Graphique sauvegardé !")

In [ ]:
# Graphique 2 : Répartition des genres de livres empruntés (camembert)
import matplotlib.pyplot as plt

genres_count = {}
for emprunt in emprunts:
    livre = livres_index[emprunt["id_livre"]]
    genre = livre["genre"]
    if genre not in genres_count:
        genres_count[genre] = 0
    genres_count[genre] += 1

plt.figure(figsize=(7, 7))
plt.pie(
    genres_count.values(),
    labels=genres_count.keys(),
    autopct="%1.0f%%",
    startangle=90
)
plt.title("Répartition des genres empruntés")
plt.tight_layout()
plt.savefig("graphique_genres.png")
plt.show()

### ✏️ Exercice 3 – Crée ton propre graphique

In [ ]:
# Crée un graphique en barres montrant le nombre d'emprunts par livre
# (titre des livres en abscisse, nombre d'emprunts en ordonnée)
# AIDE : commence par compter les emprunts par id_livre, puis retrouve les titres

# Ton code ici :


---

## 📝 Résumé

**Pourquoi plusieurs tables ?**  
Pour éviter les redondances et faciliter la mise à jour des données. Chaque information n'est stockée qu'**une seule fois**.

**Comment croiser deux tables ?**  
Grâce à un **descripteur commun** (identifiant) qui relie les tables entre elles.

**La visualisation graphique** permet de rendre les données plus lisibles et de dégager des tendances qu'un tableau brut ne montrerait pas.

---
*Prochaine séance : Le cloud, le stockage des données et l'impact environnemental du numérique*